<a href="https://colab.research.google.com/github/Boran-Sert/SoftITO_Projler/blob/master/Python/23_10_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q -U transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 74.6 MB/s eta 0:00:00


In [16]:
import torch,json,re
from transformers import AutoTokenizer, AutoModelForCausalLM

Model_adı = "Qwen/Qwen3-4B"
cihaz = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(Model_adı, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(Model_adı, device_map="auto", dtype=torch.float16 if cihaz == "cuda" else torch.float32, trust_remote_code=True)


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [22]:
def uret(mesajlar, max_yeni_token=1024, temperature=0.5, seed=42):
  """Mesaj listesini alır, modelin cevabını döndürür ve <think> kısmını temizler"""
  torch.manual_seed(seed)
  metin = tokenizer.apply_chat_template(mesajlar, tokenize=False, add_generation_prompt=True)
  tokenizasyon = tokenizer(metin, return_tensors="pt").to(model.device)

  ayar = dict(
      max_new_tokens=max_yeni_token,
      do_sample=True if temperature > 0 else False,
      temperature=temperature if temperature > 0 else None,
      pad_token_id=tokenizer.eos_token_id
  )
  if temperature > 0:
    ayar.update(top_p=0.9)

  with torch.no_grad():
    cikti = model.generate(**tokenizasyon, **ayar)

  cevap = tokenizer.decode(cikti[0][tokenizasyon["input_ids"].shape[1]:], skip_special_tokens=True).strip()

  # En sade ve doğrudan çözüm: Eğer </think> varsa sonrasını, yoksa direkt cevabı döndür.
  return cevap.split("</think>")[-1].strip() if "</think>" in cevap else cevap.strip()

def sor(kullanıcı, sistem=None, **kw):
  mesajlar = []
  if sistem:
    mesajlar.append({"role": "system", "content": sistem})
  mesajlar.append({"role": "user", "content": kullanıcı})
  return uret(mesajlar, **kw)

def karşilaştir(baslik_a,cevap_a,baslik_b,cevap_b):
    print(f":D {baslik_a}:\n{cevap_a}")
    print(f"\n:D {baslik_b}:\n{cevap_b}")
    print(f"\n" + ">:( "*5)

In [4]:
soru = "Yumurta mı tavuktan geldi, tavuk mu yumurtandan geldi ?"
karşilaştir("Direkt soru",sor(soru),
            "Rol: 8 yaşındaki çoçucağa anlatan bir öğretmen",
            sor(soru,sistem="Rol: 8 yaşındaki çoçucağa anlatan bir öğretmen"))

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


:D Direkt soru:
Bu cümlede, "Yumurta" ve "tavuk" arasındaki ilişkisi hakkında bir bilgi verirken, "tavuk"ın "yumurtan" olduğunu belirtir. Ancak, bu nedenle "yumurtan" anlamına gelmez. Bu cümlede, "tavuk"ın "yumurtan" olduğunu belirtmek için, "yumurtan" anlamına geleneğin.

Sonuç olarak, bu cümlede "yumurtan" anlamına geleneğin, ancak "tavuk"ın "yumurtan" olduğunu belirtmek için "yumurtan" anlamına geleneğin.
:D Rol: 8 yaşındaki çoçucağa anlatan bir öğretmen:
Bu cümlede, "tavuk" ve "yumurtan" arasındaki ilişkisi hakkında bir şekilde açıklamak gerekirse:

1. Yumurta: Bu bir köyün adıdır.

2. Tavuk: Bu bir köyün başkenti olan Yumurta'dır.

3. Yumurtan: Bu bir köyün başkenti olan Yumurta'dır.

4. Yumurta'yı yumurtan olarak bilinirken, yumurtanın Yumurta'ya kadar uzanan bir köydir.

5. Bu nedenle, Yumurta'nın yumurtanın Yumurta'ya kadar uzanan köyler arasında yer alıyor.

6. Ancak, bu noktada birbirine bağlı olmaz. Yumurta'ya yumurtanın Yumurta'ya kadar uzanan köyler arasında yer almakla il

In [5]:
karşilaştir(
    "belirsiz",sor("python hakkında bir şey yaz"),
    "Net ve Kısıtlı",
    sor("python programlama dilinin yeni başlayanlar için 3 avantajını yaz.\n"
        "Kurallar: \n -madde işaretli liste kullan\n"
        "- Türkçe yaz \n - Giriş veya sonuç cümlesi ekleme"),
)

:D belirsiz:
Python, Python, veya Python 3, bu bilimsel ve teknoloji sistemlerin en yaygın kullanılan bir programming語言dır. Bu programlama dilinin temel özellikleri, geliştirme süreci ve geliştirme araçları içerir.

1. Programlama dilini öğrenmek: Python, ilk olarak 1989 yılında geliştirilmiştir ve 2000'ların başında yaygın olarak kullanılmaktadır. Python'un çok çeşitli özelliklere sahiptir:

   - İmplemanlı: Python, kodunuzu işlevsel olarak yapmak için kullanılır.
   
   - Daha fazla özelleştirilmiş: Python, daha fazla özellikli ve özelleştiren bir programlama dilidir.
   
   - Çeşitli kütüphaneler: Python, birçok kütüphanedeki özelliklerle çalışır ve bu özellikler genellikle Python'den kullanılabilir.

2. Geliştirme: Python, geliştirme sürecinde önemli bir rol oynar. Python'un geliştiricileri, geliştirme sürecine ve geliştirme araçlarına ihtiyaç duyurulmuş olabileceğini belirtir.

3. Uygulamalar: Python, u
:D Net ve Kısıtlı:
1. **Yeni Başlık**: "Python Programlama Dili Başvuru" (Pyth

In [6]:
metin = ("Yapay zeka, makinelerin insan benzeri görevleri yerine getirmesini sağlayan bir bilim dalıdır."
          "Sağlık, eğitim ve ulaşım gibi pek çok alanda kullanmaktadır. Ancak etik ve güvenlik"
          "konularında dikkat olunması gerekir."
          )
prompt = f"""<metin> etiketleri arasındaki yazıyı tek cümleyle özetle

<metin>
{metin}
</metin>"""
print(sor(prompt,sistem="Sen kısa ve öz özetler yazan bir asistansın. Türkç"))

Yapay zeka, insan benzeri görevlerinin yasal olarak kabul edilmesinde önemli rol oynar. Ayrıca, etik ve güvenlik konularında önem taşır.


In [7]:
yorum = "kargo çok geç ama ürün kaliteli oldu."

zero = sor(f"Bu yorumun duygusunu söyle sadece duygu durumunu söyle:{yorum}")


mesajlar =[
    {"role":"system","content":"ürün yorumlarını sınıflandfır. Sadece şu etikeletlerden birini yaz: Olumlu, Olumsuz, Karışık"},
    {"role":"user","content":"Harika bir ürün, bir tane de kaynıma aldım. Herkese tavsiye ederim"},
    {"role":"assistant","content":"Olumlu"},
    {"role":"user","content":"kargom kaynanamın evine gitti, çok kızgınım size."},
    {"role":"assistant","content":"Karışık"},
    {"role":"user","content":"İstediğim ürün bana geldi ama bir beden küçüğü geldi"},
    {"role":"assistant","content":"Olumsuz"}


]

few =uret(mesajlar,max_yeni_token =20)
karsilaştir = ("Zero-shot",zero,"Few-shot",few)
print(karsilaştir)

('Zero-shot', 'Duyguları: \n\n1. Kargo zamanı çok kısa ve ürün kalitesini arttırdı.\n2. Ürünlerin kendi parçalarında daha iyi kalıyor.\n\nYorumunun durumu:\n\nBu durum, kargo hızının azalmasına neden olurdu. Ürünlerin kendi parçalarında daha iyi kalması, ürünün kalitesi üzerindeki değişiklikleri göz önünde bulundurarak, kargo hızını azaltmak için önemlidir. Bu durum, ürünlerin kendi parçalarında daha iyi kalmasını sağlayacak ve ürün kalitesini artırma konusunda daha fazla çaba göstermektedir.', 'Few-shot', 'Karakçık')


In [15]:
def json_cek(metin):
  """Model çıktısındaki ilk JSON  nesnesini bulup sözlüğü çevirir"""

  eslesme = re.search(r"\{.*\}",metin,re.DOTALL)
  if not eslesme:
    return None
  try:
    return json.loads(eslesme.group())
  except Json.JSONDecodeError:
    return None

sistem = """ Metin bilgi çıkaran bir asistansın. SADECE geçerli JSON döndür,
  başka hiçbir şey yazma. Şema: {"ad:" string, "yas": number, "sehir": string,
  "meslek": string} Bilgi yoksa değer olarak null kullan."""

metin = """Merhaba, ben Boran, 22 yaşındayım, İstanbul'da yaşıyorum. yapay zeka mühendisi
  olarak çalışıyorum"""
ham = sor(metin,sistem=sistem)
print("ham çıktı",ham)

veri = json_cek(ham)
print("\n Python Sözlüğü",veri)
if veri:
  print("Şehir",veri.get("sehir"))

ham çıktı Merhaba! Ben bir yapay zeka mühendis ve İstanbul'daki 22 yaşındaki bir çocuğa dair bir şekilde konuşabilirim. Size nasıl yardımcı olabilirim?

 Python Sözlüğü None


In [21]:
role_propmt = """ Sen yapay zeka mühendisliği alanında çalışan senior bir
mühendisin. Cevaplarını verirken teknik olman gerekiyor. Yapay zeka alanındaki
teknik terimleri kullan. SADECE Türkçe CEVAP VER"""

tree_of_thoughts_prompt= """ Sana verilen görevi analiz edip 3 farklı çözüm yolu
üretmen gerekli. Bu ürettiğin 3 yolu aralında avantajlarını ve dezavantajlarına
karşılaştırarak en uygun yöntemi seçmen gerekiyor. Bulduğun 3 yolu da paylaşmalısın
seçtin en uygun yolu neden seçtiğini anlatmalısın. SADECE TÜRKÇE CEVAP VER"""

role_sorusu = sor("ML hakkında neler bilmem gerekli",role_propmt)
tree_sorusu = sor("Yaptığım siteyi deploy etmek istiyorum ama para ödemek istemiyorum",tree_of_thoughts_prompt)

karsilaştir = ("Role:",role_sorusu,"Tree:",tree_sorusu)
print(karsilaştir)

('Role:', 'Yapay zeka (YŻ) ve makine öğrenimi (ML) alanında derin bir bilgiye sahip olmak, aşağıdaki teknik alanları kapsar:\n\n1. **Temel ML Kavramları**  \n   - **Üretimli (Supervised) ve Üretimli Olmayan (Unsupervised) Öğrenme**  \n     - Klasifikasyon (Sınıflandırma), regresyon, kümelenme (clustering), asimilasyon (dimensionality reduction) gibi temel görevler.  \n   - **Öğrenme Modelleri**  \n     - Lineer regresyon, karar ağaçları, support vector machines (SVM), random forest, gradient boosting (XGBoost, LightGBM), k-nearest neighbors (KNN) gibi algoritmalar.  \n   - **Öğrenme Kuramları**  \n     - Bayes teoremi, maksimum olabilirlik (MLE), maksimum olabilirlik ile en küçük kareler (MSE) gibi kavramlar.\n\n2. **Veri İşleme ve Öğrenme Süreci**  \n   - **Veri Ön İşleme (Data Preprocessing)**  \n     - Normalizasyon (min-max, z-score), standartlaştırma, veri eksikliklerinin doldurulması (imputation), kategorik değişkenlerin encode edilmesi (one-hot, label encoding).  \n   - **Özelli